In [1]:
import os
os.environ.pop("MPLBACKEND", None)

import matplotlib.pyplot as plt

In [2]:
import os
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
# or force it:
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA")

CUDA_VISIBLE_DEVICES: None
cuda available: True
device count: 1
NVIDIA GeForce RTX 3070 Laptop GPU


In [3]:
import os
SEED = 67


import gc
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
from datetime import datetime
from scipy import signal

print("torch:", torch.__version__)
print("cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("cuda device name:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

torch.manual_seed(SEED)

torch: 2.12.0+cu126
cuda build: 12.6
cuda available: True
cuda device count: 1
cuda device name: NVIDIA GeForce RTX 3070 Laptop GPU
cuda


In [4]:
from data_processing.multimodal_radar_dataset import radar_dataset_multimodal

In [5]:
sample_size= 4 
dataset= radar_dataset_multimodal("C:\\Users\\Jacobs laptop\\rainraingoaway\\data\\70km\\png", r"C:\Users\Jacobs laptop\rainraingoaway\data\environment", list_length=sample_size, total=None)
split_idx = int(0.8 * len(dataset))


train_dataset = torch.utils.data.Subset(dataset, range(0, split_idx))
test_dataset = torch.utils.data.Subset(dataset, range(split_idx, len(dataset)))

loaded 202605191430.png with shape (5, 120, 217)
loaded 202605191435.png with shape (5, 120, 217)
loaded 202605191440.png with shape (5, 120, 217)
loaded 202605191445.png with shape (5, 120, 217)
loaded 202605191450.png with shape (5, 120, 217)
loaded 202605191455.png with shape (5, 120, 217)
loaded 202605191500.png with shape (5, 120, 217)
loaded 202605191505.png with shape (5, 120, 217)
loaded 202605191510.png with shape (5, 120, 217)
loaded 202605191515.png with shape (5, 120, 217)
loaded 202605191520.png with shape (5, 120, 217)
loaded 202605191525.png with shape (5, 120, 217)
loaded 202605191530.png with shape (5, 120, 217)
loaded 202605191535.png with shape (5, 120, 217)
loaded 202605191540.png with shape (5, 120, 217)
loaded 202605191545.png with shape (5, 120, 217)
loaded 202605191550.png with shape (5, 120, 217)
loaded 202605191555.png with shape (5, 120, 217)
loaded 202605191600.png with shape (5, 120, 217)
loaded 202605191605.png with shape (5, 120, 217)
loaded 202605191610.

In [6]:
from torch.utils.data import DataLoader
BATCH_SIZE = 16 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train batches :{len(train_loader)}")
print(f"Test batches :{len(test_loader)}")

Train samples: 1545
Test samples: 387
Train batches :97
Test batches :25


In [7]:
for inputs, labels in train_loader:
    print("inputs shape:", inputs.shape)
    print("labels shape:", labels.shape)
    break

inputs shape: torch.Size([16, 3, 5, 120, 217])
labels shape: torch.Size([16, 120, 217])


In [8]:
#import model
from mode import radar_cnn
from models.convlstm import ConvLSTM

model = ConvLSTM(input_dim=5,
                 hidden_dim=64,
                 kernel_size=(3, 3),
                 num_layers=2,
                 batch_first=True,
                 bias=True)


model = model.to(device)

In [9]:
#define loss function
import torch.optim
optimizer = torch.optim.SGD(model.parameters(), lr=0.007)

pos_weight = 30.0
neg_weight = 1.0
bce_logits_loss = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([30.0]).to(device))
def weighted_regression_loss(predictions, targets):
    weights = torch.where(targets > 0, pos_weight, neg_weight)
    return ((predictions - targets) ** 2 * weights).mean()

def weighted_smooth_l1_loss(pred, target, pos_weight=30.0, neg_weight=1.0, threshold=0.1):
    """
    pred:   (B, 1, H, W), sigmoid output 0-1
    target: (B, 1, H, W), normalized target 0-1
    """
    rain_mask = target > threshold

    weights = torch.where(
        rain_mask,
        torch.tensor(pos_weight, device=target.device),
        torch.tensor(neg_weight, device=target.device)
    )

    loss = F.smooth_l1_loss(pred, target, reduction='none')
    weighted_loss = loss * weights

    return weighted_loss.mean()
def weighted_smooth_l1_loss(pred, target, pos_weight=30.0, neg_weight=1.0, threshold=0.05):
    rain_mask = target > threshold
    weights = torch.where(
        rain_mask,
        torch.tensor(pos_weight, device=target.device),
        torch.tensor(neg_weight, device=target.device)
    )
    weights = weights / weights.mean()

    loss = F.smooth_l1_loss(pred, target, reduction='none')
    return (loss * weights).mean()

def radar_intensity_loss(
    pred,
    target,
    rain_threshold=0.01,
    background_weight=0.2,
    rain_weight=2.5,
    intensity_weight=2.0,
    beta=0.05
):
    loss = F.smooth_l1_loss(
        pred,
        target,
        reduction="none",
        beta=beta
    )

    rain_mask = (target > rain_threshold).float()

    weights = background_weight + rain_mask * rain_weight + target * intensity_weight

    return (loss * weights).sum() / weights.sum().clamp_min(1.0)

loss_fn = radar_intensity_loss

In [10]:
def train_one_epoch(epoch_index, optimizer, model, loss_fn):
    running_loss = 0 
    last_loss = 0 
    for i , data in enumerate(train_loader):
        inputs, labels= data 
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        layer_output_list = model(inputs)

        outputs = layer_output_list
        
        #if labels.dim() == 3:
         #   labels = labels.unsqueeze(1)  # [batch, 120, 217] -> [batch, 1, 120, 217]
        
        #print(f"label: {labels.shape}")
        #print(f"output: {outputs.shape}")
        outputs= outputs.squeeze(1)
        #print(outputs.shape)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if i % 10 == 9:
            last_loss = running_loss / 10 # loss per batch
            print(f'  batch {i + 1} loss: {last_loss}')
            tb_x = epoch_index * len(train_loader) + i + 1
           
            running_loss = 0.
    return last_loss  

In [11]:


epoch_number = 0
val=[]
train=[]
EPOCHS = 200

best_vloss = 1000000000000000


for epoch in range(EPOCHS):
    print(f'EPOCH {epoch_number + 1}:')

    # Make sure gradient tracking is on, and do a pass over the data
    model.train(True)
    avg_train_loss = train_one_epoch(epoch_number, optimizer,model,loss_fn)


    running_vloss = 0.0
    # Set the model to evaluation mode, disabling dropout and using population
    # statistics for batch normalization.
    model.eval()

    # Disable gradient computation and reduce memory consumption.
    with torch.no_grad():
        for i, vdata in enumerate(test_loader):
            vinputs, vlabels = vdata
            vinputs = vinputs.to(device)
            vlabels = vlabels.to(device)
            layer_output_list = model(vinputs)

            voutputs = layer_output_list.squeeze(1)
            vloss = loss_fn(voutputs, vlabels.float())
            # [B, T, C, H, W] -> [B, C, H, W]
            running_vloss += vloss

    avg_val_loss = running_vloss / (i + 1)
    print(f'LOSS train {avg_train_loss} valid {avg_val_loss}')
    val.append(avg_val_loss.item())
    train.append(avg_train_loss)

    # Log the running loss averaged per batch
    # for both training and validation


    # Track best performance, and save the model's state
    if avg_val_loss < best_vloss:
        best_vloss = avg_val_loss
        model_path = f'model_best.pkl' 
        torch.save(model.state_dict(), model_path,)

    epoch_number += 1

EPOCH 1:
  batch 10 loss: 0.07240480035543442
  batch 20 loss: 0.0706153366714716
  batch 30 loss: 0.06589507451280951
  batch 40 loss: 0.07729072775691748
  batch 50 loss: 0.08043254464864731
  batch 60 loss: 0.07401759773492814
  batch 70 loss: 0.07066631782799959
  batch 80 loss: 0.06994776874780655
  batch 90 loss: 0.0872497759759426
LOSS train 0.0872497759759426 valid 0.03774971514940262
EPOCH 2:
  batch 10 loss: 0.07514473106712102
  batch 20 loss: 0.057118064258247614
  batch 30 loss: 0.07499081548303366
  batch 40 loss: 0.06703316904604435
  batch 50 loss: 0.06391199510544539
  batch 60 loss: 0.08082832973450423
  batch 70 loss: 0.08572101313620806
  batch 80 loss: 0.07823075503110885
  batch 90 loss: 0.06042943820357323
LOSS train 0.06042943820357323 valid 0.038337189704179764
EPOCH 3:
  batch 10 loss: 0.06392618771642447
  batch 20 loss: 0.07072130162268878
  batch 30 loss: 0.0700840556062758
  batch 40 loss: 0.07727656681090593
  batch 50 loss: 0.06782509032636881
  batch 60

KeyboardInterrupt: 

In [16]:
model2 = ConvLSTM(input_dim=5,
                 hidden_dim=64,
                 kernel_size=(3, 3),
                 num_layers=2,
                 batch_first=True,
                 bias=True)

model2 = model2.to(device)
model2.load_state_dict(torch.load(r"C:\Users\Jacobs laptop\rainraingoaway\src\model_best.pkl"))
print("Loaded model_best.pkl into model2")

Loaded model_best.pkl into model2


In [17]:

plt.plot(train, label="Training Loss")
plt.plot(val, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.legend()
plt.grid(True)
plt.show()

In [18]:
import matplotlib.pyplot as plt
import torch

model2.eval()

# get one batch
inputs, labels = next(iter(train_loader),4)
inputs = inputs.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model2(inputs)
sample_no=6
# pick one sample from the batch
pred = outputs[sample_no]
actual = labels[sample_no]

# if shape is [1, H, W], remove the channel dimension
if pred.dim() == 3:
    pred = pred.squeeze(0)

if actual.dim() == 3:
    actual = actual.squeeze(0)

# move to cpu and convert to numpy
pred = pred.cpu().numpy()
actual = actual.cpu().numpy()

diff = pred - actual

print(f"Prediction range: {pred.min():.4f} to {pred.max():.4f}")
print(f"Actual range: {actual.min():.4f} to {actual.max():.4f}")
print(f"Difference range: {diff.min():.4f} to {diff.max():.4f}")

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(actual, aspect='auto', cmap='viridis')
plt.title("Actual")
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(pred, aspect='auto', cmap='viridis')
plt.title("Prediction")
plt.colorbar()

plt.subplot(1, 3, 3)
plt.imshow(diff, aspect='auto', cmap='coolwarm')
plt.title("Difference")
plt.colorbar()

plt.tight_layout()
plt.show()

Prediction range: -0.0312 to 0.0740
Actual range: 0.0000 to 0.4400
Difference range: -0.4331 to 0.0740


In [19]:
import matplotlib.pyplot as plt
import torch

model2.eval()

# get one batch
inputs, labels = next(iter(train_loader),3)
inputs = inputs.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model2(inputs)
sample_no=6
# pick one sample from the batch
pred = outputs[sample_no]
actual = labels[sample_no]

# if shape is [1, H, W], remove the channel dimension
if pred.dim() == 3:
    pred = pred.squeeze(0)

if actual.dim() == 3:
    actual = actual.squeeze(0)

# move to cpu and convert to numpy
pred = pred.cpu().numpy()
actual = actual.cpu().numpy()

diff = pred - actual

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(actual, aspect='auto', cmap='viridis')
plt.title("Actual")
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(pred, aspect='auto', cmap='viridis')
plt.title("Prediction")
plt.colorbar()

plt.subplot(1, 3, 3)
plt.imshow(diff, aspect='auto', cmap='coolwarm')
plt.title("Difference")
plt.colorbar()

plt.tight_layout()
plt.show()

In [20]:
import matplotlib.pyplot as plt

model2.eval()

sample_idx = 3
sample_inputs, sample_target = test_dataset[sample_idx] if len(test_dataset) > 0 else dataset[sample_idx]

with torch.no_grad():
    sample_batch = sample_inputs.unsqueeze(0).to(device)
    sample_outputs = model2(sample_batch)
    sample_prediction = torch.sigmoid(sample_outputs)[0].cpu()

sample_inputs = sample_inputs.cpu()
sample_target = sample_target.cpu()

num_steps = sample_inputs.shape[0]
fig, axes = plt.subplots(2, max(num_steps + 1, 3), figsize=(4 * max(num_steps + 1, 3), 8))
axes = axes.reshape(2, -1)

for step in range(axes.shape[1]):
    axes[0, step].axis('off')
    axes[1, step].axis('off')

for step in range(num_steps):
    input_frame = sample_inputs[step]
    if input_frame.dim() == 3:
        input_frame = input_frame.squeeze(0)
    axes[0, step].imshow(input_frame.numpy(), aspect='auto', cmap='viridis')
    axes[0, step].set_title(f'Input t-{num_steps - step}')
    axes[0, step].axis('off')

pred_frame = sample_prediction
if pred_frame.dim() == 3:
    pred_frame = pred_frame.squeeze(0)
axes[0, num_steps].imshow(pred_frame.numpy(), aspect='auto', cmap='viridis')
axes[0, num_steps].set_title('Predicted next frame')
axes[0, num_steps].axis('off')

target_frame = sample_target
if target_frame.dim() == 3:
    target_frame = target_frame.squeeze(0)
diff_frame = pred_frame.numpy() - target_frame.numpy()
axes[1, num_steps - 1].imshow(diff_frame, aspect='auto', cmap='coolwarm')
axes[1, num_steps - 1].set_title('Prediction - truth')
axes[1, num_steps - 1].axis('off')

axes[1, num_steps].imshow(target_frame.numpy(), aspect='auto', cmap='viridis')
axes[1, num_steps].set_title('True next frame')
axes[1, num_steps].axis('off')

fig.suptitle(f'Random sample index: {sample_idx} | sequence length: {num_steps}', fontsize=14)
plt.tight_layout()
plt.show()

TypeError: Invalid shape (5, 120, 217) for image data

In [ ]:
from pathlib import Path
from datetime import datetime

save_dir = Path(r"C:\Users\Jacobs laptop\rainraingoaway\models")
save_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = save_dir / f"model_best_{timestamp}.pkl"
torch.save(model2.state_dict(), model_path)
print(f"saved to {model_path}")

saved to C:\Users\Jacobs laptop\rainraingoaway\models\model_best_20260623_104758.pkl
